# 01 Data Preparation 
### What is done here: 
 - Raw data from BEIR is downloaded and preprocessed a pandas DF
### Data Source: 

In [ ]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util
import pandas as pd
import re
from preparation_utils import (convert_beir_dbpedia_entity_v2_to_df,
                               add_wikidata_id,
                               add_wikipedia_page_id,
                               preprocess_wikiPageLinks,
                               preprocess_wikidataLinks)

In [ ]:
dataset = "dbpedia-entity"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
data_path = util.download_and_unzip(url, "../03_data/raw_data/beir_dbpedia")
corpus, queries, qrels = GenericDataLoader(data_folder="../03_data/raw_data/beir_dbpedia/dbpedia-entity").load(split="test")

# Prepare nodes

In [ ]:
mapping = []
with open("../03_data/raw_data/dbpedia/page_ids_en.ttl") as file:
    lines = file.readlines()
    for line in lines[1:-1]:
        mapping.append({
                "id": line.split(" ")[0].replace("http://dbpedia.org/resource/","dbpedia:"),
                "page_id": int(re.search(r'"\d{1,8}"', line).group()[1:-1])
            })

pd.DataFrame(mapping).to_parquet('../03_data/preprocessed_data/mapping_dbpedia_wikidata.parquet')

In [ ]:
df = convert_beir_dbpedia_entity_v2_to_df(corpus)
df.to_parquet("../03_data/preprocessed_data/nodes_beir_dbpedia.parquet")
# For reference if a wikidata id link is required
paged_df = add_wikipedia_page_id(df, pd.read_parquet('../03_data/preprocessed_data/mapping_dbpedia_wikidata.parquet'))
wikidata_df = add_wikidata_id(paged_df)

## Prepare Pagelink Relations

In [ ]:
page_links = preprocess_wikiPageLinks(df, "../03_data/raw_data/dbpedia/page_links_unredirected_en.ttl")
page_links.to_parquet("../03_data/preprocessed_data/pagelink_edges_beir_dbpedia.parquet")

## Prepare Wikidata Relations

In [ ]:
wikidata_links = preprocess_wikidataLinks(wikidata_df, "../03_data/raw_data/dbpedia/raw_unredirected_wikidata.ttl")
wikidata_links.to_parquet("../03_data/preprocessed_data/wikilink_edges_beir_dbpedia.parquet")